# TSG Consumer Partners — ML Model Training & Registry

This notebook:
1. Loads feature data from TSG_INTELLIGENCE tables
2. Trains 4 ML models with proper train/test evaluation
3. Registers all models in the **Snowflake Model Registry**
4. Verifies registered models with inference tests

**Models:**
- **Brand LTV** — RandomForestRegressor for brand lifetime value scoring
- **Churn Risk** — RandomForestClassifier for portfolio churn prediction
- **Revenue Forecast** — GradientBoostingRegressor for revenue growth prediction
- **Market Opportunity** — GradientBoostingRegressor for opportunity scoring

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.ml.registry import Registry
import snowflake.snowpark.functions as F
import pandas as pd
import numpy as np

session = get_active_session()
session.sql("USE DATABASE TSG_INTELLIGENCE").collect()
session.sql("USE SCHEMA ANALYTICS").collect()
session.sql("USE WAREHOUSE TSG_WH").collect()

registry = Registry(session=session, database_name='TSG_INTELLIGENCE', schema_name='MODELS')
print('Session initialized')
print(f'Registry: TSG_INTELLIGENCE.MODELS')

In [ ]:
%%sql -r ml_features
SELECT
    pc.COMPANY_ID,
    pc.COMPANY_NAME,
    pc.SECTOR,
    pc.INVESTMENT_AMOUNT,
    pc.EMPLOYEE_COUNT,
    AVG(rd.REVENUE) AS avg_revenue,
    AVG(rd.GROSS_MARGIN) AS avg_gross_margin,
    AVG(rd.EBITDA_MARGIN) AS avg_ebitda_margin,
    AVG(rd.REVENUE_GROWTH_YOY) AS avg_revenue_growth,
    AVG(bm.BRAND_AWARENESS) AS avg_brand_awareness,
    AVG(bm.BRAND_SENTIMENT) AS avg_brand_sentiment,
    AVG(bm.NET_PROMOTER_SCORE) AS avg_nps,
    AVG(bm.BRAND_EQUITY_INDEX) AS avg_brand_equity,
    AVG(bm.CUSTOMER_SAT_SCORE) AS avg_csat,
    AVG(bm.SOCIAL_ENGAGEMENT) AS avg_social_engagement,
    AVG(da.CONVERSION_RATE) AS avg_conversion_rate,
    AVG(da.LTV_CAC_RATIO) AS avg_ltv_cac,
    AVG(da.PAID_ROAS) AS avg_roas,
    AVG(om.INVENTORY_TURNOVER) AS avg_inventory_turnover,
    AVG(om.SUPPLY_CHAIN_SCORE) AS avg_supply_chain,
    AVG(om.CASH_CONVERSION) AS avg_cash_conversion,
    MAX(mr.MARKET_GROWTH_RATE) AS market_growth_rate,
    MAX(mr.TAM_USD) AS tam_usd
FROM TSG_INTELLIGENCE.ANALYTICS.PORTFOLIO_COMPANIES pc
LEFT JOIN TSG_INTELLIGENCE.ANALYTICS.REVENUE_DATA rd ON pc.COMPANY_ID = rd.COMPANY_ID AND rd.FISCAL_YEAR >= 2024
LEFT JOIN TSG_INTELLIGENCE.ANALYTICS.BRAND_METRICS bm ON pc.COMPANY_ID = bm.COMPANY_ID AND bm.METRIC_DATE >= '2024-01-01'
LEFT JOIN TSG_INTELLIGENCE.ANALYTICS.DIGITAL_ANALYTICS da ON pc.COMPANY_ID = da.COMPANY_ID AND da.METRIC_DATE >= '2024-01-01'
LEFT JOIN TSG_INTELLIGENCE.ANALYTICS.OPERATIONS_METRICS om ON pc.COMPANY_ID = om.COMPANY_ID AND om.METRIC_DATE >= '2024-01-01'
LEFT JOIN TSG_INTELLIGENCE.ANALYTICS.MARKET_RESEARCH mr ON pc.COMPANY_ID = mr.COMPANY_ID
WHERE pc.STATUS = 'Active'
GROUP BY pc.COMPANY_ID, pc.COMPANY_NAME, pc.SECTOR, pc.INVESTMENT_AMOUNT, pc.EMPLOYEE_COUNT
ORDER BY pc.COMPANY_ID

In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.preprocessing import LabelEncoder

df = ml_features.copy()
df = df.fillna(0)

le = LabelEncoder()
df['SECTOR_ENCODED'] = le.fit_transform(df['SECTOR'])

feature_cols = ['INVESTMENT_AMOUNT', 'EMPLOYEE_COUNT', 'AVG_REVENUE', 'AVG_GROSS_MARGIN',
                'AVG_EBITDA_MARGIN', 'AVG_REVENUE_GROWTH', 'AVG_BRAND_EQUITY',
                'AVG_NPS', 'AVG_SOCIAL_ENGAGEMENT', 'AVG_CONVERSION_RATE',
                'AVG_LTV_CAC', 'MARKET_GROWTH_RATE', 'SECTOR_ENCODED']

X = df[feature_cols]
print(f'Dataset shape: {df.shape}')
print(f'Companies: {len(df)}')
print(f'Features: {len(feature_cols)}')
df[['COMPANY_NAME', 'SECTOR'] + feature_cols[:5]].head(12)

In [ ]:
ltv_target = (df['AVG_REVENUE'] / 1000000 * 0.3 +
              df['AVG_BRAND_EQUITY'] * 0.25 +
              df['AVG_REVENUE_GROWTH'] * 0.2 +
              df['AVG_NPS'] * 0.15 +
              df['MARKET_GROWTH_RATE'] * 0.1)

X_train, X_test, y_train, y_test = train_test_split(X, ltv_target, test_size=0.25, random_state=42)

ltv_model = RandomForestRegressor(n_estimators=100, random_state=42, max_depth=5)
ltv_model.fit(X_train, y_train)

y_pred_train = ltv_model.predict(X_train)
y_pred_test = ltv_model.predict(X_test)

print('=== Brand LTV Model — RandomForestRegressor ===')
print(f'\nTrain Metrics:')
print(f'  R²:  {r2_score(y_train, y_pred_train):.4f}')
print(f'  MAE: {mean_absolute_error(y_train, y_pred_train):.4f}')
print(f'  RMSE: {np.sqrt(mean_squared_error(y_train, y_pred_train)):.4f}')
print(f'\nTest Metrics:')
print(f'  R²:  {r2_score(y_test, y_pred_test):.4f}')
print(f'  MAE: {mean_absolute_error(y_test, y_pred_test):.4f}')
print(f'  RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_test)):.4f}')

cv_scores = cross_val_score(ltv_model, X, ltv_target, cv=min(3, len(X)), scoring='r2')
print(f'\nCross-Validated R²: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')

ltv_model.fit(X, ltv_target)
df['LTV_PREDICTED'] = ltv_model.predict(X)

print(f'\nTop Features:')
for col, imp in sorted(zip(feature_cols, ltv_model.feature_importances_), key=lambda x: -x[1])[:5]:
    print(f'  {col}: {imp:.4f}')

ltv_metrics = {
    'r2_test': round(float(r2_score(y_test, y_pred_test)), 4),
    'mae_test': round(float(mean_absolute_error(y_test, y_pred_test)), 4),
    'rmse_test': round(float(np.sqrt(mean_squared_error(y_test, y_pred_test))), 4),
    'cv_r2_mean': round(float(cv_scores.mean()), 4)
}

In [ ]:
churn_target = (df['AVG_NPS'] < df['AVG_NPS'].median()).astype(int)

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X, churn_target, test_size=0.25, random_state=42, stratify=churn_target)

churn_model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=4)
churn_model.fit(X_train_c, y_train_c)

y_pred_train_c = churn_model.predict(X_train_c)
y_pred_test_c = churn_model.predict(X_test_c)
y_prob_test_c = churn_model.predict_proba(X_test_c)[:, 1]

print('=== Churn Risk Model — RandomForestClassifier ===')
print(f'\nTrain Metrics:')
print(f'  Accuracy:  {accuracy_score(y_train_c, y_pred_train_c):.4f}')
print(f'  Precision: {precision_score(y_train_c, y_pred_train_c, zero_division=0):.4f}')
print(f'  Recall:    {recall_score(y_train_c, y_pred_train_c, zero_division=0):.4f}')
print(f'  F1:        {f1_score(y_train_c, y_pred_train_c, zero_division=0):.4f}')
print(f'\nTest Metrics:')
print(f'  Accuracy:  {accuracy_score(y_test_c, y_pred_test_c):.4f}')
print(f'  Precision: {precision_score(y_test_c, y_pred_test_c, zero_division=0):.4f}')
print(f'  Recall:    {recall_score(y_test_c, y_pred_test_c, zero_division=0):.4f}')
print(f'  F1:        {f1_score(y_test_c, y_pred_test_c, zero_division=0):.4f}')

try:
    auc = roc_auc_score(y_test_c, y_prob_test_c)
    print(f'  AUC:       {auc:.4f}')
except ValueError:
    auc = 0.0
    print(f'  AUC:       N/A (single class in test set)')

cv_scores_c = cross_val_score(churn_model, X, churn_target, cv=min(3, len(X)), scoring='accuracy')
print(f'\nCross-Validated Accuracy: {cv_scores_c.mean():.4f} (+/- {cv_scores_c.std():.4f})')

churn_model.fit(X, churn_target)
df['CHURN_RISK_PREDICTED'] = churn_model.predict_proba(X)[:, 1]

print(f'\nTop Features:')
for col, imp in sorted(zip(feature_cols, churn_model.feature_importances_), key=lambda x: -x[1])[:5]:
    print(f'  {col}: {imp:.4f}')

churn_metrics = {
    'accuracy_test': round(float(accuracy_score(y_test_c, y_pred_test_c)), 4),
    'f1_test': round(float(f1_score(y_test_c, y_pred_test_c, zero_division=0)), 4),
    'auc_test': round(float(auc), 4),
    'cv_accuracy_mean': round(float(cv_scores_c.mean()), 4)
}

In [ ]:
forecast_target = df['AVG_REVENUE_GROWTH']

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(X, forecast_target, test_size=0.25, random_state=42)

forecast_model = GradientBoostingRegressor(n_estimators=100, random_state=42, max_depth=3, learning_rate=0.1)
forecast_model.fit(X_train_f, y_train_f)

y_pred_train_f = forecast_model.predict(X_train_f)
y_pred_test_f = forecast_model.predict(X_test_f)

print('=== Revenue Forecast Model — GradientBoostingRegressor ===')
print(f'\nTrain Metrics:')
print(f'  R²:  {r2_score(y_train_f, y_pred_train_f):.4f}')
print(f'  MAE: {mean_absolute_error(y_train_f, y_pred_train_f):.4f}')
print(f'  RMSE: {np.sqrt(mean_squared_error(y_train_f, y_pred_train_f)):.4f}')
print(f'\nTest Metrics:')
print(f'  R²:  {r2_score(y_test_f, y_pred_test_f):.4f}')
print(f'  MAE: {mean_absolute_error(y_test_f, y_pred_test_f):.4f}')
print(f'  RMSE: {np.sqrt(mean_squared_error(y_test_f, y_pred_test_f)):.4f}')

cv_scores_f = cross_val_score(forecast_model, X, forecast_target, cv=min(3, len(X)), scoring='r2')
print(f'\nCross-Validated R²: {cv_scores_f.mean():.4f} (+/- {cv_scores_f.std():.4f})')

forecast_model.fit(X, forecast_target)
df['GROWTH_FORECAST'] = forecast_model.predict(X)

print(f'\nTop Features:')
for col, imp in sorted(zip(feature_cols, forecast_model.feature_importances_), key=lambda x: -x[1])[:5]:
    print(f'  {col}: {imp:.4f}')

forecast_metrics = {
    'r2_test': round(float(r2_score(y_test_f, y_pred_test_f)), 4),
    'mae_test': round(float(mean_absolute_error(y_test_f, y_pred_test_f)), 4),
    'rmse_test': round(float(np.sqrt(mean_squared_error(y_test_f, y_pred_test_f))), 4),
    'cv_r2_mean': round(float(cv_scores_f.mean()), 4)
}

In [ ]:
opportunity_target = (df['MARKET_GROWTH_RATE'] * 3 +
                      np.log10(df['TAM_USD'].clip(lower=1)) * 2 +
                      df['AVG_CONVERSION_RATE'] * 5 +
                      df['AVG_LTV_CAC'] * 3)

X_train_o, X_test_o, y_train_o, y_test_o = train_test_split(X, opportunity_target, test_size=0.25, random_state=42)

opp_model = GradientBoostingRegressor(n_estimators=100, random_state=42, max_depth=3)
opp_model.fit(X_train_o, y_train_o)

y_pred_train_o = opp_model.predict(X_train_o)
y_pred_test_o = opp_model.predict(X_test_o)

print('=== Market Opportunity Model — GradientBoostingRegressor ===')
print(f'\nTrain Metrics:')
print(f'  R²:  {r2_score(y_train_o, y_pred_train_o):.4f}')
print(f'  MAE: {mean_absolute_error(y_train_o, y_pred_train_o):.4f}')
print(f'  RMSE: {np.sqrt(mean_squared_error(y_train_o, y_pred_train_o)):.4f}')
print(f'\nTest Metrics:')
print(f'  R²:  {r2_score(y_test_o, y_pred_test_o):.4f}')
print(f'  MAE: {mean_absolute_error(y_test_o, y_pred_test_o):.4f}')
print(f'  RMSE: {np.sqrt(mean_squared_error(y_test_o, y_pred_test_o)):.4f}')

cv_scores_o = cross_val_score(opp_model, X, opportunity_target, cv=min(3, len(X)), scoring='r2')
print(f'\nCross-Validated R²: {cv_scores_o.mean():.4f} (+/- {cv_scores_o.std():.4f})')

opp_model.fit(X, opportunity_target)
df['OPPORTUNITY_SCORE'] = opp_model.predict(X)

print(f'\nTop Features:')
for col, imp in sorted(zip(feature_cols, opp_model.feature_importances_), key=lambda x: -x[1])[:5]:
    print(f'  {col}: {imp:.4f}')

opp_metrics = {
    'r2_test': round(float(r2_score(y_test_o, y_pred_test_o)), 4),
    'mae_test': round(float(mean_absolute_error(y_test_o, y_pred_test_o)), 4),
    'rmse_test': round(float(np.sqrt(mean_squared_error(y_test_o, y_pred_test_o))), 4),
    'cv_r2_mean': round(float(cv_scores_o.mean()), 4)
}

In [ ]:
results = df[['COMPANY_NAME', 'SECTOR', 'LTV_PREDICTED', 'CHURN_RISK_PREDICTED', 'GROWTH_FORECAST', 'OPPORTUNITY_SCORE']].copy()
results.columns = ['Company', 'Sector', 'LTV Score', 'Churn Risk', 'Growth Forecast %', 'Opportunity Score']
results = results.sort_values('LTV Score', ascending=False)

print('=== TSG Consumer Partners — ML Model Evaluation Summary ===')
print(f'\nBrand LTV Model:        R²={ltv_metrics["cv_r2_mean"]}, Test MAE={ltv_metrics["mae_test"]}')
print(f'Churn Risk Model:       Acc={churn_metrics["cv_accuracy_mean"]}, Test F1={churn_metrics["f1_test"]}, AUC={churn_metrics["auc_test"]}')
print(f'Revenue Forecast Model: R²={forecast_metrics["cv_r2_mean"]}, Test MAE={forecast_metrics["mae_test"]}')
print(f'Opportunity Model:      R²={opp_metrics["cv_r2_mean"]}, Test MAE={opp_metrics["mae_test"]}')
print(f'\n{results.to_string(index=False)}')

In [ ]:
sample_input = X.head(3)

mv_ltv = registry.log_model(
    model=ltv_model,
    model_name='TSG_BRAND_LTV_MODEL',
    version_name='v2',
    sample_input_data=sample_input,
    metrics=ltv_metrics,
    target_platforms=['WAREHOUSE'],
    comment='Brand Lifetime Value prediction model — RandomForestRegressor trained on portfolio company features'
)
print(f'Registered: TSG_BRAND_LTV_MODEL v2')
print(f'  Metrics: {ltv_metrics}')

mv_churn = registry.log_model(
    model=churn_model,
    model_name='TSG_CHURN_RISK_MODEL',
    version_name='v2',
    sample_input_data=sample_input,
    metrics=churn_metrics,
    target_platforms=['WAREHOUSE'],
    comment='Portfolio Churn Risk classification model — RandomForestClassifier trained on brand health and financial features'
)
print(f'Registered: TSG_CHURN_RISK_MODEL v2')
print(f'  Metrics: {churn_metrics}')

mv_forecast = registry.log_model(
    model=forecast_model,
    model_name='TSG_REVENUE_FORECAST_MODEL',
    version_name='v2',
    sample_input_data=sample_input,
    metrics=forecast_metrics,
    target_platforms=['WAREHOUSE'],
    comment='Revenue Growth Forecast model — GradientBoostingRegressor trained on historical revenue and market data'
)
print(f'Registered: TSG_REVENUE_FORECAST_MODEL v2')
print(f'  Metrics: {forecast_metrics}')

mv_opp = registry.log_model(
    model=opp_model,
    model_name='TSG_MARKET_OPPORTUNITY_MODEL',
    version_name='v2',
    sample_input_data=sample_input,
    metrics=opp_metrics,
    target_platforms=['WAREHOUSE'],
    comment='Market Opportunity scoring model — GradientBoostingRegressor trained on TAM, competitive position, and digital metrics'
)
print(f'Registered: TSG_MARKET_OPPORTUNITY_MODEL v2')
print(f'  Metrics: {opp_metrics}')

print(f'\nAll 4 models registered in TSG_INTELLIGENCE.MODELS with target_platform=WAREHOUSE')

In [ ]:
print('=== Registered Models in TSG_INTELLIGENCE.MODELS ===')
models = registry.show_models()
print(models.to_string(index=False))

print('\n=== Inference Test — Brand LTV Model ===')
test_input = X.head(3)
ltv_preds = mv_ltv.run(test_input, function_name='predict')
print(ltv_preds)

print('\n=== Inference Test — Churn Risk Model ===')
churn_preds = mv_churn.run(test_input, function_name='predict')
print(churn_preds)

print('\n=== Inference Test — Revenue Forecast Model ===')
forecast_preds = mv_forecast.run(test_input, function_name='predict')
print(forecast_preds)

print('\n=== Inference Test — Market Opportunity Model ===')
opp_preds = mv_opp.run(test_input, function_name='predict')
print(opp_preds)

print('\nAll 4 models verified successfully!')